In [1]:
# Configuration: List of directory paths to compare
directories = [
    "../results/gpt_4.1-qwen25-32b-prod/",
    "../results/gpt_4.1-surgical_adapter_v26_qwen3_8b_step_0_strict_format_check-prod/",
    "../results/gpt_4.1-surgical_adapter_v27_qwen3_8b_step_0_strict_format_check-prod/",
    "../results/gpt_4.1-surgical_adapter_v28_qwen3_8b_step_0_strict_format_check-prod/",
    "../results/gpt_4.1-surgical_adapter_v29_qwen3_8b_step_0_strict_format_check-prod/",
]

In [2]:
import json
import os
import plotly.graph_objects as go
from pathlib import Path

In [3]:
def load_data(json_file):
    with open(json_file, 'r') as f:
        data = json.load(f)
    return data

def calculate_difficulty(runs):
    """
    Every run has a list of dict.
    Each dict has a task_id and a reward field.
    We want to calculate the difficulty of the task.
    Difficulty is basically how many times the reward is 1 for each task.
    Returns a list of len(tasks)
    """
    ret = {}  # task_id -> count_of_1s
    for run in runs:
        for task in run:
            if task['task_id'] not in ret:
                ret[task['task_id']] = 0
            if task['reward'] == 1:
                ret[task['task_id']] += 1
    return list(ret.values())

def reduce_task_wise_difficulty(task_wise_difficulty, n_runs):
    """
    Reduce the task-wise difficulty to a list of length n_runs+1 (0 to n_runs)
    """
    ret = [0] * (n_runs + 1)
    for x in task_wise_difficulty:
        ret[x] += 1
    return ret

def process_directory(results_dir):
    """
    Process a single directory and return its difficulty distribution and number of runs.
    """
    files = sorted(os.listdir(results_dir))
    runs = []
    for file in files:
        if file.endswith('.json'):
            data = load_data(os.path.join(results_dir, file))
            runs.append(data)
    
    if not runs:
        print(f"Warning: No JSON files found in {results_dir}")
        return [], 0
    
    task_wise_difficulty = calculate_difficulty(runs)
    difficulty = reduce_task_wise_difficulty(task_wise_difficulty, len(runs))
    return difficulty, len(runs)

In [4]:
# Process all directories
difficulties_by_dir = {}

for directory in directories:
    difficulty, n_runs = process_directory(directory)
    if difficulty:
        dir_name = os.path.basename(directory.rstrip('/'))
        difficulties_by_dir[dir_name] = difficulty
        print(f"Processed {dir_name}: {len(difficulty)} difficulty levels, {n_runs} runs")

Processed gpt_4.1-qwen25-32b-prod: 6 difficulty levels, 5 runs
Processed gpt_4.1-surgical_adapter_v26_qwen3_8b_step_0_strict_format_check-prod: 6 difficulty levels, 5 runs
Processed gpt_4.1-surgical_adapter_v27_qwen3_8b_step_0_strict_format_check-prod: 6 difficulty levels, 5 runs
Processed gpt_4.1-surgical_adapter_v28_qwen3_8b_step_0_strict_format_check-prod: 6 difficulty levels, 5 runs
Processed gpt_4.1-surgical_adapter_v29_qwen3_8b_step_0_strict_format_check-prod: 6 difficulty levels, 5 runs


In [9]:
# Create interactive plotly figure
fig = go.Figure()

# Determine the max length for consistent x-axis
max_len = max(len(difficulty) for difficulty in difficulties_by_dir.values())
n_runs = max_len - 1

# Add a trace for each directory
for dir_name, difficulty in difficulties_by_dir.items():
    x_positions = list(range(len(difficulty)))
    x_labels = [f'{i}/{n_runs}' for i in x_positions]
    
    fig.add_trace(go.Scatter(
        x=x_labels,
        y=difficulty,
        mode='lines+markers',
        name=dir_name,
        marker=dict(size=8),
        line=dict(width=2),
        hovertemplate='<b>%{fullData.name}</b><br>' +
                      'Score: %{x}<br>' +
                      'Number of tasks: %{y}<br>' +
                      '<extra></extra>'
    ))

# Update layout
fig.update_layout(
    title='Difficulty Distribution Comparison',
    xaxis_title='Score',
    yaxis_title='Number of tasks',
    hovermode='closest',
    width=1200,
    height=800,
    template='plotly_white',
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.1,
        xanchor="center",
        x=0.5
    )
)

fig.show()